# LEVELS Benchmark — Level 1: Atomic

Single DSI/API call from natural language. Each task is scored by 7 binary checks: tool, args, path, count, keywords, forbidden keywords and nonempty response. `score` is the fraction of checks passed, and `passed` means all 7 passed. A judge ensemble also scores efficiency against the human `reference_path`.

| Level | Notebook |
|---|---|
| 1 Atomic | `LEVELS_1_atomic.ipynb` |
| 2 Workflow | `LEVELS_2_workflow.ipynb` |
| 3 Discovery | `LEVELS_3_discovery.ipynb` |
| Calibration (RQ4) | `LEVELS_4_calibration.ipynb` |
| Prompt phrasing | `LEVELS_5_prompt_variants.ipynb` |
| Cross-level RQ summary | `LEVELS_6_summary.ipynb` |

### Data Management Categories

1. Search: external/domain knowledge and literature
2. Data Discovery: locating datasets, files, or records in DSI repositories
3. Data Preparation: cleaning, validating, standardizing scientific data
4. Metadata Generation: structured documentation and provenance
5. Analysis & Summarization: interpreting, aggregating, synthesizing data

### MCP mode

To benchmark through MCP (atomic tools served by `levels_mcp_server.py`): run `python levels_mcp_server.py` in a terminal, then pass `use_mcp=True` to `init(...)`. Inspector: `npx @modelcontextprotocol/inspector` → Streamable HTTP → `http://127.0.0.1:8000/mcp`.

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [1]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [2]:
from levels_bench import init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import LEVEL1_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

Working directory: /Users/hhn/lanl/dsi/tools/ai_search
Run id: 20261022_084429 | models: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0'] | judges: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0']
Workers: 4 | turn timeout: 180s | skip_network: False


In [3]:
reports.task_table(LEVEL1_TASKS)

task_id,level,category,turns,network,variants,query
str,i64,str,i64,bool,i64,str
"""L1_load_ocean11""",1,"""Data Access""",1,false,5,"""Load the ocean_11_datasets.db …"
"""L1_load_nif_direct""",1,"""Data Access""",1,false,5,"""Load the nif.db database."""
"""L1_load_poisson""",1,"""Data Access""",1,false,5,"""Load the poisson.db database."""
"""L1_search_implosion""",1,"""Data Discovery""",1,false,5,"""Do you have any implosion data…"
"""L1_download_nif_sim""",1,"""Data Retrieval""",1,true,5,"""Download the simulation data f…"
…,…,…,…,…,…,…
"""L1_extract_originating_org""",1,"""Metadata Query""",1,false,5,"""What organization is listed fo…"
"""L1_extract_physics_theme_count""",1,"""Metadata Query""",1,false,5,"""How many datasets are categori…"
"""L1_nif_domain_extent""",1,"""Data Exploration""",1,false,5,"""What are the spatial bounds of…"


## Run

In [4]:
l1_results = flatten(run_benchmark(LEVEL1_TASKS, "level1_atomic"))
# To analyze an earlier run without re-running it:
# l1_results = load_results("level1_atomic")
l1_df = results_to_dataframe(l1_results)
l1_df

level1_atomic: 13 tasks x 2 models = 26 runs (26 to run, 4 workers)
[ 1/26] PASS    aws-gov.gpt-5.4 | L1_load_ocean11  score=1.00  6s   (elapsed 0m08s, ~3m29s left)
[ 2/26] PASS    aws-gov.gpt-5.4 | L1_load_nif_direct  score=1.00  7s   (elapsed 0m08s, ~1m42s left)
[ 3/26] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L1_load_nif_direct  score=1.00  10s   (elapsed 0m11s, ~1m31s left)
[ 4/26] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L1_load_ocean11  score=1.00  10s   (elapsed 0m12s, ~1m06s left)
[ 5/26] PASS    aws-gov.gpt-5.4 | L1_load_poisson  score=1.00  5s   (elapsed 0m13s, ~0m54s left)
[ 6/26] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L1_load_poisson  score=1.00  9s   (elapsed 0m17s, ~0m57s left)
[ 7/26] PASS    aws-gov.gpt-5.4 | L1_search_implosion  score=1.00  6s   (elapsed 0m18s, ~0m49s left)
[ 8/26] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L1_search_implosion  score=1.00  11s   (elapsed 0m22s, ~0m51s left)
[ 9/26] PASS    aws-gov.gpt-5.4 | L

model,task_id,query,prompt_style,base_task_id,level,category,status,passed,score,elapsed_sec,wall_sec,tokens,tools,response,notes,expects_abstention,hallucination_detected,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category,check_tool_ok,check_args_ok,check_path_ok,check_quant_ok,check_keyword_ok,check_forbidden_ok,check_nonempty_ok,judge_efficiency_score,judge_efficiency_reasoning
str,str,str,str,str,i64,str,str,bool,f64,f64,f64,i64,str,str,str,bool,bool,i64,i64,i64,str,bool,bool,bool,bool,bool,bool,bool,f64,str
"""aws-gov.gpt-5.4""","""L1_load_ocean11""","""Load the ocean_11_datasets.db …","""standard""","""L1_load_ocean11""",1,"""Data Access""","""PASS""",true,1.0,3.0,6.32,2301,"""load_dsi_tool""","""Loaded `/Users/hhn/lanl/dsi/to…","""""",false,false,0,0,0,null,true,true,true,true,true,true,true,0.125,"""[aws-gov.gpt-5.4] The referenc…"
"""aws-gov.gpt-5.4""","""L1_load_nif_direct""","""Load the nif.db database.""","""standard""","""L1_load_nif_direct""",1,"""Data Access""","""PASS""",true,1.0,3.2,6.51,1931,"""load_dsi_tool""","""Loaded `nif.db`. Current data…","""""",false,false,0,0,0,null,true,true,true,true,true,true,true,0.4,"""[aws-gov.gpt-5.4] The agent su…"
"""aws-gov.gpt-5.4""","""L1_load_poisson""","""Load the poisson.db database.""","""standard""","""L1_load_poisson""",1,"""Data Access""","""PASS""",true,1.0,2.3,4.71,2032,"""load_dsi_tool""","""Loaded `poisson.db`. Current …","""""",false,false,0,0,0,null,true,true,true,true,true,true,true,0.4,"""[aws-gov.gpt-5.4] The agent su…"
"""aws-gov.gpt-5.4""","""L1_search_implosion""","""Do you have any implosion data…","""standard""","""L1_search_implosion""",1,"""Data Discovery""","""PASS""",true,1.0,3.1,6.21,2369,"""query_dsi_tool""","""Yes — I found one implosion-re…","""""",false,false,0,0,0,null,true,true,true,true,true,true,true,0.61,"""[aws-gov.gpt-5.4] The agent co…"
"""aws-gov.gpt-5.4""","""L1_download_nif_sim""","""Download the simulation data f…","""standard""","""L1_download_nif_sim""",1,"""Data Retrieval""","""PASS""",true,1.0,15.8,19.2,4886,"""query_dsi_tool, query_dsi_tool…","""Downloaded successfully: `/Us…","""""",false,false,1,1,0,null,true,true,true,true,true,true,true,0.27,"""[aws-gov.gpt-5.4] The agent ev…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""anthropic.claude-sonnet-4-5-20…","""L1_extract_originating_org""","""What organization is listed fo…","""standard""","""L1_extract_originating_org""",1,"""Metadata Query""","""PASS""",true,1.0,4.3,7.06,3024,"""query_dsi_tool""","""The organization listed for th…","""""",false,false,0,0,0,null,true,true,true,true,true,true,true,1.0,"""[aws-gov.gpt-5.4] The agent ex…"
"""anthropic.claude-sonnet-4-5-20…","""L1_extract_physics_theme_count""","""How many datasets are categori…","""standard""","""L1_extract_physics_theme_count""",1,"""Metadata Query""","""PASS""",true,1.0,4.4,7.66,3011,"""query_dsi_tool""","""There are **5 datasets** categ…","""""",false,false,0,0,0,null,true,true,true,true,true,true,true,0.95,"""[aws-gov.gpt-5.4] The agent us…"
"""anthropic.claude-sonnet-4-5-20…","""L1_nif_domain_extent""","""What are the spatial bounds of…","""standard""","""L1_nif_domain_extent""",1,"""Data Exploration""","""PASS""",true,1.0,27.5,31.49,7859,"""query_dsi_tool, query_dsi_tool…","""Based on the information from …","""""",false,false,0,0,0,null,true,true,true,true,true,true,true,0.135,"""[aws-gov.gpt-5.4] The agent to…"


## Results

In [5]:
reports.model_summary(l1_df)

### Overall summary by model

model,pass_rate,avg_score,avg_efficiency,avg_tokens,avg_elapsed_sec,n_tasks,n_errors
str,f64,f64,f64,f64,f64,u32,u32
"""anthropic.claude-sonnet-4-5-20…",1.0,1.0,0.44,4407.31,12.24,13,0
"""aws-gov.gpt-5.4""",0.92,0.99,0.54,2531.77,4.4,13,0


### Pass rate by model and category

model,Data Access,Data Discovery,Data Retrieval,Metadata Query,Data Exploration
str,f64,f64,f64,f64,f64
"""aws-gov.gpt-5.4""",1.0,1.0,1.0,0.67,1.0
"""anthropic.claude-sonnet-4-5-20…",1.0,1.0,1.0,1.0,1.0


### Pass/fail matrix by task and model

task_id,aws-gov.gpt-5.4,anthropic.claude-sonnet-4-5-20250929-v1:0
str,str,str
"""L1_load_ocean11""","""PASS""","""PASS"""
"""L1_load_nif_direct""","""PASS""","""PASS"""
"""L1_load_poisson""","""PASS""","""PASS"""
"""L1_search_implosion""","""PASS""","""PASS"""
"""L1_download_nif_sim""","""PASS""","""PASS"""
…,…,…
"""L1_extract_originating_org""","""PASS""","""PASS"""
"""L1_extract_physics_theme_count""","""PASS""","""PASS"""
"""L1_nif_domain_extent""","""PASS""","""PASS"""


In [6]:
reports.failures(l1_df)

### Non-passing runs (1)

model,task_id,status,score,notes,failure_category
str,str,str,f64,str,str
"""aws-gov.gpt-5.4""","""L1_owner_field""","""FAIL""",0.857,"""Response missing keywords: ['Joshua', 'Paul', 'Sauppe']""","""MODEL_REASONING"""


In [7]:
reports.timing_breakdown(l1_results)

### Where does the time go?

#### Top-level phases (real wall-clock, no double counting)

phase_group,total_sec,mean_sec,count
str,f64,f64,u32
"""step""",216.259,8.318,26
"""score""",81.283,0.782,104


#### By model x top-level phase group (total seconds)

model,step,score
str,f64,f64
"""anthropic.claude-sonnet-4-5-20…",159.17,40.27
"""aws-gov.gpt-5.4""",57.09,41.01


#### Nested inside steps (already counted above)

phase,total_sec,mean_sec,count
str,f64,f64,u32
"""llm_call""",212.05,2.28,93
"""tool:download_file_tool""",3.591,0.598,6
"""tool:query_dsi_tool""",0.302,0.006,52
"""tool:load_dsi_tool""",0.058,0.004,14


## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [8]:
# reports.transcript(l1_results, "L1_load_ocean11")
# tr = debug_task(get_task("L1_load_ocean11"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")